# Master Dataset Creation

In [7]:
import pandas as pd
import re

In [4]:
human_df = pd.read_csv("../data/human_dataset.csv")
ai_df = pd.read_csv("../data/ai_generated_zeroshot.csv")

print(human_df.shape)
print(ai_df.shape)

(600, 5)
(600, 8)


In [5]:
# colums check
print(human_df.columns)
print(ai_df.columns)


Index(['source_id', 'source', 'text', 'label', 'generation_type'], dtype='str')
Index(['source_id', 'source', 'text', 'label', 'generation_type', 'topic',
       'latency_sec', 'error'],
      dtype='str')


In [6]:
# check missing

print(human_df.isna().sum())
print(ai_df.isna().sum())

source_id          0
source             0
text               0
label              0
generation_type    0
dtype: int64
source_id            0
source               0
text                 0
label                0
generation_type      0
topic                0
latency_sec          0
error              600
dtype: int64


In [8]:
# remove non-ASCII characters from AI text

def clean_text(text):
    if pd.isna(text):
        return text
    return re.sub(r'[^\x00-\x7F]+', '', text)

ai_df["text"] = ai_df["text"].apply(clean_text)

In [9]:
# remove broken rows
human_df = human_df[human_df["text"].str.strip() != ""]
ai_df = ai_df[ai_df["text"].str.strip() != ""]

In [11]:
# check labels
print(human_df["label"].value_counts())
print(human_df["generation_type"].value_counts())
print(human_df["source"].value_counts())

print(ai_df["label"].value_counts())
print(ai_df["generation_type"].value_counts())
print(ai_df["source"].value_counts())

label
human    600
Name: count, dtype: int64
generation_type
human    600
Name: count, dtype: int64
source
articles.csv        200
yelp.csv            200
essays_texts.csv    200
Name: count, dtype: int64
label
ai    600
Name: count, dtype: int64
generation_type
zero_shot    600
Name: count, dtype: int64
source
ai_essay     200
ai_review    200
ai_news      200
Name: count, dtype: int64


In [12]:
# drop AI only columns
ai_df = ai_df[["source_id", "source", "text", "label", "generation_type"]]

In [13]:
# merge data
df = pd.concat([human_df, ai_df], ignore_index=True)

In [15]:
# shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df["id"] = df.index # add id column

In [16]:
# reorder columns
df = df[["id", "text", "label","generation_type", "source", "source_id"]]

In [17]:
print("\nFinal Shape")
print(df.shape)

print("\n-Label Counts")
print(df["label"].value_counts())

print("\n-Source Counts")
print(df["source"].value_counts())


Final Shape
(1200, 6)

-Label Counts
label
ai       600
human    600
Name: count, dtype: int64

-Source Counts
source
ai_essay            200
articles.csv        200
ai_review           200
ai_news             200
essays_texts.csv    200
yelp.csv            200
Name: count, dtype: int64


In [18]:
df.head(5)

,id,text,label,generation_type,source,source_id
0,0,Have you ever pondered the evolution of indiv...,ai,zero_shot,ai_essay,ai_essay_21
1,1,"Dear Reader,\n\nLet us embark on an explorati...",ai,zero_shot,ai_essay,ai_essay_9
2,2,London: Crude prices fell Thursday awaiting th...,human,human,articles.csv,articles_324
3,3,ISLAMABAD: Pakistan captain Inamul Haq is opti...,human,human,articles.csv,articles_1592
4,4,Title: A Sunny Day Revival at Culver's After ...,ai,zero_shot,ai_review,ai_review_12


In [19]:
# save
df.to_csv("../data/master_0_dataset.csv", index=False)